# LFM2.5-1.2B-Instruct-Uncensored -> Hindi + HinglishThree chained sessions on Kaggle 2x T4. Each trains on the previous one's pushedoutput, so a VM reset never costs more than 5 minutes.| session | data | teaches ||---|---|---|| 1 | `train_s1_hinglish` | romanized Hinglish || 2 | `train_s2_hindi` | Devanagari Hindi || 3 | `train_s3_english` | English anchor / anti-forgetting |Edit only `HF_USER` below. Then run cells in order, top to bottom.**Never paste a token into this notebook - it is public.** Use the Kaggle secretnamed `HF_TOKEN` (right sidebar -> Add-Ons -> Secrets), then restart the session.

In [ ]:
# ---------------------------------------------------------------- run configSESSION   = 1HF_USER   = "kumarsujitroy"        # <-- the ONLY thing you must editDATA_FILE = {1: "train_s1_hinglish.jsonl",             2: "train_s2_hindi.jsonl",             3: "train_s3_english.jsonl"}[SESSION]OUT_REPO  = f"{HF_USER}/lfm25-1.2b-bilingual-s{SESSION}"BASE_REPO = ("zaakirio/LFM2.5-1.2B-Instruct-Uncensored" if SESSION == 1             else f"{HF_USER}/lfm25-1.2b-bilingual-s{SESSION - 1}")MAX_STEPS     = {1: 690, 2: 400, 3: 620}[SESSION]LEARNING_RATE = {1: 2e-4, 2: 1.5e-4, 3: 5e-5}[SESSION]   # steps down: chained runs driftCHECKPOINT_EVERY_S = 300assert SESSION in (1, 2, 3)print(f"session {SESSION}  data={DATA_FILE}")print(f"  base  {BASE_REPO}")print(f"  out   {OUT_REPO}")

## 0. Login`push_to_hub` is the whole point: it is the only thing that survives a Kaggle VMreset. Without a token every checkpoint push fails and the run is lost.**One-time setup:** right sidebar -> **Add-Ons** -> **Secrets** -> **+ New Secret**- Label `HF_USER`'s HF token, i.e. `hf_...`, with **`write`** scope- **tick Notebook Access**- **restart the session** - Kaggle injects secrets only at session startThe notebook is public, so there is deliberately no paste slot anywhere in it.

In [ ]:
import osfrom huggingface_hub import login, whoami     # whoami is a function, not a CLI commandTOKEN = ""try:    from kaggle_secrets import UserSecretsClient    TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""except Exception:    passTOKEN = (TOKEN or os.environ.get("HF_TOKEN", "")).strip()if not TOKEN:    raise RuntimeError(        "No HF token. Add-Ons (right sidebar) -> Secrets -> + New Secret\n"        "  Label: HF_TOKEN   Value: your hf_ write token\n"        "Then RESTART the session.")if not TOKEN.startswith("hf_"):    raise ValueError(f"Token does not look like an HF token: {TOKEN[:6]}...")login(token=TOKEN, add_to_git_credential=False)os.environ["HF_TOKEN"] = TOKENprint("logged in as:", whoami(token=TOKEN).get("name"))

In [ ]:
# unsloth MUST be imported before transformers anywhere in this notebook, or it# warns and skips its patches. Every later `from transformers import ...` is# then safe.!pip install -q unsloth!pip install -q "transformers==4.57.3"          # 4.57 is what unsloth is tested against!pip install -q --no-deps trl==0.22.2!pip install -q datasets==4.3.0 accelerate peft bitsandbytes hf_transferimport unslothfrom unsloth import FastLanguageModelprint("unsloth imported first - OK")

## Crash-proof checkpointing**A local save on Kaggle is not a backup.** `/kaggle/working` is destroyed when theVM resets, taking every `save_steps` checkpoint with it. The only durable store isthe Hub, so this pushes the adapter on a wall-clock timer rather than a step count.If the session dies at minute 38 you resume from minute 35.**Watch for these lines.** If they never appear, the push is failing silently and areset destroys the run.

In [ ]:
import timefrom transformers import TrainerCallbackclass TimeCheckpoint(TrainerCallback):    """Push the adapter + tokenizer to the Hub every `every_s` wall-clock seconds."""    def __init__(self, model, repo, every_s=300, tokenizer=None):        self.model, self.repo, self.every = model, repo, every_s        self.tok, self.n, self.last = tokenizer, 0, time.time()    def on_step_end(self, args, state, control, **kw):        if time.time() - self.last >= self.every:            self.n += 1            tag = f"ckpt-s{state.global_step // 100 * 100:05d}"            self.model.push_to_hub(self.repo, save_name=tag)            if self.tok is not None:                # both, or the checkpoint is only half-recoverable                self.tok.push_to_hub(self.repo, save_name=tag)            print(f"[{time.strftime('%H:%M:%S')}] pushed {tag} @ step {state.global_step}",                  flush=True)            self.last = time.time()        return control

## 1. Build the training dataSelf-contained, runs here on Kaggle. Writes all three session files into`/kaggle/working/`. Set `BUILD_DATA = False` to skip and use the repo's copies.**Sources, one per bucket because no single dataset is good at all three:**| session | source | why ||---|---|---|| s1 | Sujalvc AAG_anchored | conversational backbone, 68-word replies || s1 | arena + casual | real humans typing / natural register || s1 | COMI-LINGUA RH | 3-expert-annotated romanized (EMNLP 2025) || s2 | smangrul Devanagari + zicsx | conversational + clean pool || s2 | COMI-LINGUA DH | 3-expert-annotated Devanagari || s3 | dolly-15k | human-written English anchor |**Read the `median-reply-words` line before training.** A median under ~20 meansterse fragments rather than conversation - that is the failure mode that produceda model answering `kya haal hai?` with `Kya har baahar hai`. Validation loss doesNOT catch it, because short predictable output scores a *lower* loss.

In [ ]:
# Self-contained data build on Kaggle.BUILD_DATA = True# Every source that measured well gets used. Source count is not a constraint -# each session trains independently and the downloads stream.#   s1  Sujalvc + arena + casual + COMI-LINGUA   (romanized Hinglish)#   s2  smangrul + zicsx + COMI-LINGUA           (Devanagari Hindi)#   s3  smangrul English-bearing rowsHINDI_TARGET    = 6000HINGLISH_TARGET = 8500ENGLISH_TARGET  = 4000if not BUILD_DATA:    print("BUILD_DATA=False - the next cell will use the repo's jsonl")else:    import csv, json, os, random, re, urllib.request    def grab(url, dest):        os.makedirs(os.path.dirname(dest), exist_ok=True)        if not os.path.exists(dest):            with urllib.request.urlopen(url, timeout=900) as r, open(dest, "wb") as f:                f.write(r.read())        return dest    def stream_csv(url, dest, i_c, o_c, cap):        """Arena/casual are big remote CSVs - stream them, never hold 1M rows in RAM."""        p = grab(url, dest)        rows, hdr = [], None        with open(p, "rb") as f:                 # one open, binary, decode per line            for raw in f:                line = raw.decode("utf-8", "replace").rstrip("\r\n")                if not line:                    continue                if hdr is None:                    hdr = next(csv.reader([line]))                    continue                rec = dict(zip(hdr, next(csv.reader([line]))))                a, b = (rec.get(i_c) or "").strip(), (rec.get(o_c) or "").strip()                if a and b:                    rows.append((a, b))                    if len(rows) >= cap:                        break        return rows    CODE_FENCE = re.compile(r"```")    # Translated/curated data contains the ORIGINAL model refusing. Training on those    # teaches the uncensored model to refuse - the opposite of the goal.    REFUSAL = re.compile(        r"\b(i'?m sorry|i am sorry|i apolog|sorry, but|"        r"i (?:can'?t|cannot|can not|won'?t|will not)\b|"        r"i'?m (?:not able|unable|afraid)\b|i am (?:not able|unable)\b|"        r"as an ai\b|as a language model\b|i must (?:decline|refuse)\b)\b", re.I)    MARATHI = re.compile(r"(?<![\u0900-\u097F])(आहे|आणि|मला|तुम्ही|काय|मध्ये|साठी|"                         r"करण्यासाठी|छे|पण|नाही)(?![\u0900-\u097F])")    FOREIGN = {k: re.compile(v) for k, v in {        "Bengali": r"[\u0980-\u09FF]", "Gujarati": r"[\u0A00-\u0A7F]",        "Tamil": r"[\u0A80-\u0AFF]",   "Telugu": r"[\u0B00-\u0B7F]",        "Kannada": r"[\u0B80-\u0BFF]", "Malayalam": r"[\u0D00-\u0D7F]",        "Arabic": r"[\u0600-\u06FF]"}.items()}    DEV = re.compile(r"[\u0900-\u097F]")    EN_ONLY = re.compile(r"\b(the|and|is|are|of|to|with|for|that|this|you|your|"                         r"from|have|will|can|not)\b", re.I)    # 620 chars * 1.45 tok/char + ~15 template ~= 915 tokens, inside MAX_SEQ_LENGTH=1024    MAX_CHARS = 620    MIN_WORDS = 12          # <- the check that would have caught the 8-word failure    # allow_dev: True = require Devanagari in both, False = require none anywhere,    # None = require BOTH fields agree, "any" = no script constraint at all.    def clean(u, a, allow_dev, min_words=MIN_WORDS):        if not u or not a or CODE_FENCE.search(u) or CODE_FENCE.search(a):            return None        if REFUSAL.search(u) or REFUSAL.search(a) or MARATHI.search(u) or MARATHI.search(a):            return None        for rx in FOREIGN.values():            if rx.search(u) or rx.search(a): return None        if allow_dev is None and bool(DEV.search(u)) != bool(DEV.search(a)):            return None                                   # one side Devanagari, other not        if allow_dev == "any":            pass                                         # mixed is the POINT here        if allow_dev not in (None, "any") and bool(DEV.search(u) or DEV.search(a)) != allow_dev:            return None        if len(u) + len(a) > MAX_CHARS: return None        if len(a.split()) < min_words: return None        # terse/fragmentary answer        if len(u.split()) < 3: return None        return u, a    random.seed(3407)    out = {"hinglish": [], "hindi": [], "english": []}    prov = {}    # ---- s1: Sujalvc, AAG_anchored only --------------------------------------    # The 379 GeminiFlash rows are skipped: their replies leak English    # ("SSY is highly targeted debt scheme for girl child education").    try:        raw = grab("https://huggingface.co/datasets/Sujalvc/hinglish-instruct-dataset/"                   "resolve/main/hinglish_chat_chunk.jsonl",                   "/kaggle/working/data_cache/sujalvc.jsonl")        n = 0        for line in open(raw, encoding="utf-8"):            try: r = json.loads(line)            except Exception: continue            if r.get("source") != "AAG_anchored": continue            c = clean(r.get("instruction", ""), r.get("output", ""), allow_dev=False)            if c:                out["hinglish"].append({"instruction": c[0], "response": c[1],                                        "src": "sujalvc", "category": "AAG"})                n += 1        prov["sujalvc"] = n    except Exception as e:        print(f"  sujalvc failed: {type(e).__name__}: {e}")    # ---- s1: real human Hinglish prompts from chatbot arena -------------------    try:        rows = stream_csv("https://huggingface.co/datasets/one-thing/"                          "chatbot_arena_conversations_hinglish/resolve/main/data.csv",                          "/kaggle/working/data_cache/arena.csv",                          "user_hinglish", "assistant_hinglish", 4000)        n = 0        for u, a in rows:            c = clean(u, a, allow_dev=False, min_words=6)            if c: out["hinglish"].append({"instruction": c[0], "response": c[1],                                          "src": "arena", "category": "arena"}); n += 1        prov["arena"] = n    except Exception as e:        print(f"  arena failed: {type(e).__name__}: {e}")    # ---- s1: natural everyday register ---------------------------------------    try:        rows = stream_csv("https://huggingface.co/datasets/Abhishekcr448/"                          "Hinglish-Everyday-Conversations-1M/resolve/main/"                          "hinglish_conversations.csv",                          "/kaggle/working/data_cache/casual.csv", "input", "output", 3000)        n = 0        for u, a in rows:            c = clean(u, a, allow_dev=False, min_words=3)            if c: out["hinglish"].append({"instruction": c[0], "response": c[1],                                          "src": "casual", "category": "casual"}); n += 1        prov["casual"] = n    except Exception as e:        print(f"  casual failed: {type(e).__name__}: {e}")    # ---- s2: smangrul Devanagari + zicsx filler -------------------------------    try:        raw = grab("https://huggingface.co/datasets/smangrul/hindi_instruct_v1/"                   "resolve/main/data/train-00000-of-00001.parquet",                   "/kaggle/working/data_cache/smangrul.parquet")        import pandas as pd        df = pd.read_parquet(raw)        n = 0        for msgs, cat in zip(df["messages"], df["category"]):            try:                u = next(m["content"] for m in msgs if m["role"] == "user")                a = next(m["content"] for m in msgs if m["role"] == "assistant")            except (StopIteration, TypeError): continue            if not (DEV.search(u) and DEV.search(a)): continue      # Devanagari only            c = clean(u, a, allow_dev=True, min_words=10)            if c: out["hindi"].append({"instruction": c[0], "response": c[1],                                       "src": "smangrul", "category": str(cat)}); n += 1        prov["smangrul_deva"] = n    except Exception as e:        print(f"  smangrul failed: {type(e).__name__}: {e}")    try:   # Devanagari filler - 512k pool, clean, 0.00% Marathi        raw = grab("https://huggingface.co/datasets/zicsx/indic-align-hindi/resolve/main/"                   "data/train-00000-of-00026-a3979045df3b161a.parquet",                   "/kaggle/working/data_cache/zicsx0.parquet")        import pandas as pd        df = pd.read_parquet(raw)        n = 0        for arr in df["interactions"]:            try:                t = list(arr)[0]; u, a = str(t[0]), str(t[1])            except Exception: continue            c = clean(u, a, allow_dev=True, min_words=12)            if c: out["hindi"].append({"instruction": c[0], "response": c[1],                                       "src": "zicsx", "category": "zicsx"}); n += 1            if len(out["hindi"]) >= HINDI_TARGET * 2: break        prov["zicsx"] = n    except Exception as e:        print(f"  zicsx failed: {type(e).__name__}: {e}")    # ---- COMI-LINGUA: 3-expert-annotated, CC-BY-4.0, EMNLP-2025 ---------------    # Each MT row is triple-parallel: Sentences (code-mixed source),    # Annotator_1_RH_translation (romanized Hindi), Annotator_1_DH_translation    # (Devanagari Hindi). This is the ONLY human-expert-annotated Hindi source in    # the whole search - everything else is machine-translated or LLM-synthesised.    # Register is news/journalism at ~20 words a sentence, so it supplements the    # conversational sources rather than replacing them.    try:        import csv as _csv        p = grab("https://huggingface.co/datasets/LingoIITGN/COMI-LINGUA/resolve/main/MT_train.csv",                 "/kaggle/working/data_cache/comi_MT.csv")        nr = nd = 0        with open(p, encoding="utf-8", errors="replace") as f:            for r in _csv.DictReader(f):                src_s = (r.get("Sentences") or "").strip()                rh = (r.get("Annotator_1_RH_translation") or "").strip()                dh = (r.get("Annotator_1_DH_translation") or "").strip()                # s1: code-mixed Devanagari prompt -> romanized Hindi reply                # source is Devanagari, reply is romanized -> allow_dev=None                # (mixed) then require the REPLY itself to carry no Devanagari.                # Passing allow_dev=False asked for no Devanagari in EITHER field                # and rejected all 24,692 rows, which is why comi_rh came out 0.                if src_s and rh and nr < HINGLISH_TARGET and not DEV.search(rh):                    q = clean(src_s, rh, allow_dev="any", min_words=12)                    if q:                        out["hinglish"].append({"instruction": q[0], "response": q[1],                                                "src": "comi_rh", "category": "expert"}); nr += 1                # s2: same source -> Devanagari Hindi reply                if src_s and dh and nd < HINDI_TARGET:                    q = clean(src_s, dh, allow_dev=True, min_words=12)                    if q:                        out["hindi"].append({"instruction": q[0], "response": q[1],                                              "src": "comi_dh", "category": "expert"}); nd += 1        prov["comi_rh"] = nr        prov["comi_dh"] = nd    except Exception as e:        print(f"  comi-lingua failed: {type(e).__name__}: {e}")    # ---- s3: English anchor ---------------------------------------------------    # NOT smangrul. Its romanized rows are Hinglish with English loanwords, so    # filtering them for "English" harvested 511 rows of more Hinglish - an anchor    # made of the very thing it is supposed to balance. dolly-15k is genuinely    # human-written English instruction data.    try:        p = grab("https://huggingface.co/datasets/databricks/databricks-dolly-15k/"                 "resolve/main/databricks-dolly-15k.jsonl",                 "/kaggle/working/data_cache/dolly.jsonl")        n = 0        for line in open(p, encoding="utf-8"):            try: r = json.loads(line)            except Exception: continue            q = clean((r.get("instruction") or "").strip(),                      (r.get("response") or "").strip(),                      allow_dev=False, min_words=12)            if q:                out["english"].append({"instruction": q[0], "response": q[1],                                       "src": "dolly", "category": "english"}); n += 1        prov["dolly"] = n    except Exception as e:        print(f"  dolly failed: {type(e).__name__}: {e}")    # ---- mix, write, and REPORT THE THING THAT MATTERS -----------------------    def dump(name, data, n_target, quotas=None):        """Exact per-source quotas, NOT shuffle-then-cap.        Shuffling the pool and taking the first N silently diluted the scarce good        sources: the hindi pool was 18,000, we took a random 6,000, and smangrul        (conversational) fell from 2,185 to 751 while 6,569 zicsx rows were thrown        away. Shortage of a good source is a fact to surface, not to average away.        """        if quotas:            by = {}            for r in data:                by.setdefault(r["src"], []).append(r)            sel, short = [], []            for s_name, want in quotas.items():                have = by.get(s_name, [])                random.shuffle(have)                sel.extend(have[:want])                if len(have) < want:                    short.append(f"{s_name} {len(have)}/{want}")            if short:                print(f"  !! {name} short of: {', '.join(short)}")        else:            random.shuffle(data)            sel = data[:n_target]        random.shuffle(sel)        p = f"/kaggle/working/{name}"        with open(p, "w", encoding="utf-8") as f:            for r in sel:                f.write(json.dumps(r, ensure_ascii=False) + "\n")        return sel    print("\n  source            rows kept")    for k, v in prov.items():        print(f"  {k:17s} {v:>8,}")    targets = {"s1_hinglish": ("hinglish", HINGLISH_TARGET),               "s2_hindi": ("hindi", HINDI_TARGET),               "s3_english": ("english", ENGLISH_TARGET)}    print("\n  session        rows   median-reply-words  sources")    quotas = {        "s1_hinglish": {"sujalvc": 4500, "arena": 1500, "casual": 1000, "comi_rh": 1500},        "s2_hindi":    {"smangrul": 6000, "comi_dh": 2000, "zicsx": 3000},        "s3_english":  {"dolly": ENGLISH_TARGET},    }    for sess, (b, t) in targets.items():        sel = dump(f"train_{sess}.jsonl", out[b], t, quotas.get(sess))        mw = sorted(len(r["response"].split()) for r in sel)        med = mw[len(mw)//2] if mw else 0        srcs = {}        for r in sel: srcs[r["src"]] = srcs.get(r["src"], 0) + 1        print(f"  {sess:13s} {len(sel):>6,}   {med:>10}          {srcs}")        if med < 20:            print(f"  !! {sess} median reply is {med} words - that is the terse-fragment "                  f"failure mode, not chat. Check the sources above.")    # script + contamination audit    print("\n--- audit ---")    for b, data in out.items():        if not data: continue        dev = sum(1 for r in data if DEV.search(r["instruction"] + r["response"]))        mar = sum(1 for r in data if MARATHI.search(r["instruction"] + r["response"]))        eng = sum(1 for r in data if len(EN_ONLY.findall(r["response"])) >= 6)        print(f"  {b:9s} n={len(data):>6,}  devanagari={dev:>6,}  marathi={mar}  english_reply={eng}")

In [ ]:
import glob, json# 0. built in the previous cell, 1. attached Kaggle Dataset, 2. public repoCANDIDATES = ([f"/kaggle/working/{DATA_FILE}"]              + [f"/kaggle/input/hinglish-sft/{DATA_FILE}"]              + glob.glob(f"/kaggle/input/*/{DATA_FILE}"))DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)if DATA_PATH is None:    url = ("https://raw.githubusercontent.com/SujitRoy/hindi-finetune/main/"           + DATA_FILE)    print(f"not built or attached - fetching {url}")    import urllib.request    with urllib.request.urlopen(url, timeout=180) as r:        payload = r.read()    DATA_PATH = f"/kaggle/working/{DATA_FILE}"    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)    with open(DATA_PATH, "wb") as f:        f.write(payload)    print(f"  fetched {len(payload):,} bytes")rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]srcs = {s: sum(1 for r in rows if r["src"] == s) for s in {r["src"] for r in rows}}print(len(rows), "rows |", srcs)dev = sum(1 for r in rows if any("\u0900" <= c <= "\u097F"                                for c in r["instruction"] + r["response"]))print(f"{dev} rows contain Devanagari")

## 1b. Teacher probe (optional, ~15 min)Every Hindi instruction corpus on the Hub is machine-translated or LLM-synthesised.The escape is to **generate our own**: human-written English instructions (dolly) witha larger model answering in Hindi, so the content is human and the language is native.**Why this cell fetches files with `urllib` instead of `from_pretrained(repo)`:**the Hub's *metadata API* returns 404 from Kaggle after heavy traffic - all fourteachers failed, including Qwen2.5-7B which had loaded minutes earlier - while the*file CDN* kept working, the data build in the same session pulled 270 MB through it.So we download every file ourselves and load from a local directory, which nevertouches the metadata API at all.Verified from this environment: all 8 files for Qwen3-8B fetch, 5 weight shards,16.4 GB total.

In [ ]:
# Optional. Set RUN_PROBE = False to skip.RUN_PROBE = TrueMODEL = "Qwen/Qwen3-8B"          # or "unsloth/Llama-3.2-3B-Instruct" (6.4 GB, ~2.5x faster)MAX_NEW = 200HIN_SYS = ("Aap ek helpful assistant ho. User ke sawaal ka jawab POORI tarah se, saaf "           "aur natural Devanagari Hindi mein likho. English se seedha translation mat "           "karo - waise likho jaise ek native Hindi speaker likhta hai. Markdown ya "           "bullet points use mat karo, plain paragraph likho.")PROMPTS = [    "Explain why the sky appears blue.",    "What is the difference between RAM and storage?",    "Why do onions make people cry?",    "My laptop fan is very loud. What should I check?",    "How can I improve my sleep quality?",    "Give me a birthday gift idea for my girlfriend.",    "kya haal hai bhai?",]SMALL_FILES = ["config.json", "generation_config.json", "tokenizer.json",               "tokenizer_config.json", "vocab.json", "merges.txt",               "added_tokens.json", "special_tokens_map.json"]def fetch_repo(repo, dest):    """Pull every file of `repo` into `dest` using plain urllib - no Hub metadata API,    which is what 404s from Kaggle after a few hundred MB of traffic."""    import os, json as _j, urllib.request, urllib.error    def get(fn):        url = f"https://huggingface.co/{repo}/resolve/main/{fn}"        with urllib.request.urlopen(url, timeout=600) as r:            return r.read()    os.makedirs(dest, exist_ok=True)    have = set(os.listdir(dest))    idx_name = "model.safetensors.index.json"    if idx_name not in have:        open(os.path.join(dest, idx_name), "wb").write(get(idx_name))        have.add(idx_name)    need = list(SMALL_FILES)    shards = set()    try:        wm = _j.load(open(os.path.join(dest, idx_name)))["weight_map"]        shards = set(wm.values())    except Exception:        need.append("model.safetensors")          # single-shard repo    need += sorted(shards)    total = 0    for fn in need:        dst = os.path.join(dest, fn)        if fn in have and os.path.getsize(dst) > 0:            continue        try:            body = get(fn)        except urllib.error.HTTPError as e:            print(f"  skip {fn} ({e.code})")            continue        with open(dst, "wb") as f:            f.write(body)        total += len(body)        print(f"  {fn:34s} {len(body):>12,}", flush=True)    print(f"  fetched {total/1e9:.1f} GB this run -> {dest}")    return destif not RUN_PROBE:    print("RUN_PROBE=False - skipping")else:    import re, time, torch    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig    # /tmp, same as the generation cell. Putting 16.4 GB of Qwen weights in    # /kaggle/working left 0.9 GB free and killed the GGUF export at the end of a    # training run, even though /tmp had over a terabyte.    import shutil as _sh    _sh.rmtree("/kaggle/working/teacher", ignore_errors=True)    DEST = "/tmp/teacher"    print(f"fetching {MODEL} ...", flush=True)    try:        fetch_repo(MODEL, DEST)    except Exception as e:        raise RuntimeError(f"urllib fetch failed: {type(e).__name__}: {e}") from e    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",                             bnb_4bit_compute_dtype=torch.bfloat16)    # local path: transformers reads the dir directly, so the Hub API is never queried    tk = AutoTokenizer.from_pretrained(DEST)    md = AutoModelForCausalLM.from_pretrained(DEST, quantization_config=bnb,                                             device_map="auto").eval()    print(f"\nloaded {MODEL} from {DEST}\n")    @torch.no_grad()    def gen(q):        msgs = [{"role": "system", "content": HIN_SYS}, {"role": "user", "content": q}]        try:   # Qwen3 emits a <think> block by default; we want just the answer            ids = tk.apply_chat_template(msgs, add_generation_prompt=True,                                         enable_thinking=False, return_tensors="pt")        except TypeError:            ids = tk.apply_chat_template(msgs, add_generation_prompt=True,                                         return_tensors="pt")        ids = ids.to(md.device)        o = md.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,                        pad_token_id=tk.pad_token_id or tk.eos_token_id)        return tk.decode(o[0][ids.shape[1]:], skip_special_tokens=True).strip()    DEV = re.compile(r"[\u0900-\u097F]")    for q in PROMPTS:        t0 = time.time()        r = gen(q)        dev = len(DEV.findall(r)) / max(1, len(r))        print(f"\nQ: {q}\nA: {r[:400]}\n   ({time.time()-t0:.1f}s  devanagari={dev:.0%}  "              f"words={len(r.split())})")    print("\n" + "=" * 70)    print("JUDGE the Hindi yourself, the script cannot:")    print("  NATIVE      -> reordering, idioms, no calque of English word order")    print("  TRANSLATION -> English order rendered in Devanagari characters")    print("  HALLUCINATED-> wrong physics/facts (Qwen2.5-7B called sky-blue ice)")    print("  TERSE       -> under 30 words. Ours collapses to 4-word fragments on")    print("                short prompts; anything the teacher does there is a win.")    print("Only proceed to bulk generation on a clean NATIVE sweep.")

## 1c. Teacher generation (optional, hours)Every Hindi instruction corpus on the Hub is machine-translated or LLM-synthesised, sothe student learns the register of those translations rather than the language. Theteacher writes the surface form correctly - measured, not assumed: asked to translatewell-formed Hindi into romanized Hinglish it produced `Agar vah kalaakaar hai, to ekchhotaa chit ... diya ja sakta hai jo uski ruchi ko dikhaye` with correct verbagreement. Asked to *generate* Hindi freely, the same model does worse. So we let itgenerate, and filter hard.**The filter is the deliverable, not the prompt.** The teacher's 7/7 probe answers werealready clean; the risk is a bad row slipping into the mix. Four rejects, and thelength floor is the important one: the student collapses below 30 words, the teacherwrites 37-49, so a 30-word floor passes real rows and kills the collapse at the source.**Where this runs.** chdir to /tmp before the teacher is fetched. Last time the 16.4 GBof Qwen weights landed in /kaggle/working and left 0.9 GB, which killed the GGUF exportat the end of training. /tmp has over a terabyte and the weights are disposable.

In [ ]:
# Optional. Set RUN_GEN = False to skip.RUN_GEN   = TrueTEACHER   = "Qwen/Qwen3-8B"          # 16.4 GB. "unsloth/Llama-3.2-3B-Instruct" is 6.4 GB and ~2.5x faster.N_PER_PASS = 800                     # 800 Hindi + 800 Hinglish ~= 10.7 h at the measured 24 s/answerMAX_NEW   = 220PUSH_EVERY_S = 300                   # same 5-minute rule as training: Kaggle storage is not durableOUT_JSONL = "/tmp/teacher_gen.jsonl"if not RUN_GEN:    print("RUN_GEN=False - skipping")else:    import os, re, csv, time, json, hashlib, shutil, urllib.request    os.chdir("/tmp")                    # teacher + output live here, never in /kaggle/working    shutil.rmtree("/kaggle/working/teacher", ignore_errors=True)    print(f"cwd {os.getcwd()}  {shutil.disk_usage('/tmp').free/2**30:.0f} GB free")    # ---- one system prompt per surface; the student needs both    DEV_SYS = ("Aap ek helpful assistant ho. User ke sawaal ka jawab POORI tarah se, saaf "               "aur natural Devanagari Hindi mein likho. English se seedha translation mat "               "karo - waise likho jaise ek native Hindi speaker likhta hai. Markdown ya "               "bullet points use mat karo, plain paragraph likho. "               "Jawab kam se kam 30 shabd ka hona chahiye.")    HING_SYS = ("Aap ek helpful assistant ho. User Hinglish mein likhta hai (romanized Hindi "                "+ English mix). Aap bhi isi tarah romanized Hinglish mein jawab do. Poora aur "                "sahi jawab do, kam se kam 30 shabd. Varanak ke liye 'hai' aur 'hain' ka sahi "                "prayog karo, 'h' ya 'kr' jaise shortcut mat likho. Markdown ya bullets use mat karo.")    # ---- rejects. Four rules; the length floor is the one that matters.    BAD = re.compile(r"(ai assistant|ai model|as an ai|मैं एक एआई|"                     r"\bh\b(?!\w)|\bkr\b|\bkro\b)", re.I)    REFU = re.compile(r"(i am sorry|i cannot|i can't|maine pucha|mujhe nahi pata|"                      r"i'm not able|as an ai)", re.I)    def accept(resp, want_dev):        """Return (ok, reason). The student collapses below 30 words; the teacher writes        37-49, so a 30 floor passes real rows and removes the collapse at the source."""        w = len(resp.split())        if w < 30:                          return False, "short"        if w > 160:                         return False, "ramble"        if BAD.search(resp):                return False, "shorthand"        if REFU.search(resp):               return False, "refusal"        # Degeneration, independent of which teacher produced it. The student emitted        # "...मंसिस्ट्रा मंसिस्ट्रा मंसिस्ट्रा (Rivers)..." and a teacher loses the        # thread the same way. A global unique-token ratio MISSES this - the sentence        # still has plenty of distinct words, it just stutters in one place - so this        # looks for a repeated trigram instead, which is the actual signature.        toks = resp.split()        tri = [tuple(toks[j:j + 3]) for j in range(len(toks) - 2)]        if tri and len(set(tri)) / len(tri) < 0.92:            return False, "repetitive"        dev = sum("\u0900" <= c <= "\u097F" for c in resp) / max(1, len(resp))        if want_dev and dev < 0.55:         return False, "not_devanagari"        if not want_dev and dev > 0.35:     return False, "went_devanagari"        return True, ""    # ---- instructions: dolly, human-written English. Content human, language native.    # The repo ships ONE file and it is jsonl, not parquet - a parquet URL 404s.    # The data-build cell already cached it; reuse that rather than fetching twice.    CACHED = "/kaggle/working/data_cache/dolly.jsonl"    DOLLY_URL = ("https://huggingface.co/datasets/databricks/databricks-dolly-15k/"                 "resolve/main/databricks-dolly-15k.jsonl")    dl = CACHED if os.path.exists(CACHED) else "/tmp/dolly.jsonl"    if not os.path.exists(dl):        with urllib.request.urlopen(DOLLY_URL, timeout=900) as r, open(dl, "wb") as f:            f.write(r.read())    print(f"dolly: {dl}")    instrs = []    for line in open(dl, encoding="utf-8"):        try: q = json.loads(line)["instruction"].strip()        except Exception: continue        if not q: continue        w = len(q.split())        # 3-40 words, and not a closed-question that wants a one-word answer.        # A yes/no prompt teaches a yes/no reply, which is the collapse we are fixing.        if not (3 <= w <= 40): continue        if q.lower().rstrip(" ?.").split()[-1] in ("or", "and", "is", "are", "the", "a", "an"): continue        if re.match(r"^(is|are|was|were|do|does|did|can|could|will|would|should|has|have|had)\b",                    q, re.I) and w <= 6:            continue        instrs.append(q)    print(f"dolly: {len(instrs)} usable instructions")    # ---- resumable: a row already on disk is never regenerated    done = set()    if os.path.exists(OUT_JSONL):        for line in open(OUT_JSONL, encoding="utf-8"):            try:                r = json.loads(line)                done.add(r["key"])            except Exception:                pass        print(f"resuming: {len(done)} rows already generated")    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig    DEST = "/tmp/teacher"    fetch_repo(TEACHER, DEST)                       # from the probe cell    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",                             bnb_4bit_compute_dtype=torch.bfloat16)    tk = AutoTokenizer.from_pretrained(DEST)    md = AutoModelForCausalLM.from_pretrained(DEST, quantization_config=bnb,                                             device_map="auto").eval()    @torch.no_grad()    def gen(q, sys_p):        msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": q}]        try:            ids = tk.apply_chat_template(msgs, add_generation_prompt=True,                                         enable_thinking=False, return_tensors="pt")        except TypeError:            ids = tk.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")        ids = ids.to(md.device)        o = md.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,                        pad_token_id=tk.pad_token_id or tk.eos_token_id)        return tk.decode(o[0][ids.shape[1]:], skip_special_tokens=True).strip()    def push():        """Kaggle local storage is not durable. Push on a wall clock, not a row count."""        from huggingface_hub import HfApi        HfApi().upload_file(path_or_fileobj=OUT_JSONL, path_in_repo="teacher_gen.jsonl",                            repo_id=OUT_REPO, commit_message=f"teacher gen {int(time.time())}")    os.makedirs("/tmp", exist_ok=True)    kept = {"hindi": 0, "hinglish": 0}    dropped = {}    last_push, t_start = time.time(), time.time()    fh = open(OUT_JSONL, "a", encoding="utf-8")    for want_dev, sys_p, tag in ((True, DEV_SYS, "hindi"), (False, HING_SYS, "hinglish")):        for q in instrs:            if kept[tag] >= N_PER_PASS:                break            key = hashlib.sha1(f"{tag}|{q}".encode()).hexdigest()[:16]            if key in done:                continue            try:                a = gen(q, sys_p)            except Exception as e:                print(f"  gen error {type(e).__name__} - skipping", flush=True)                continue            ok, why = accept(a, want_dev)            if ok:                fh.write(json.dumps({"instruction": q, "response": a, "src": "teacher",                                     "lang": tag, "key": key}, ensure_ascii=False) + "\n")                fh.flush()                kept[tag] += 1                if kept[tag] % 10 == 0:                    el = time.time() - t_start                    print(f"[{time.strftime('%H:%M:%S')}] {tag} {kept[tag]}/{N_PER_PASS}  "                          f"{el/60:.0f}m  {el/max(1,sum(kept.values())):.0f}s/row  "                          f"drop={dropped}", flush=True)            else:                dropped[why] = dropped.get(why, 0) + 1            if time.time() - last_push >= PUSH_EVERY_S:                push(); last_push = time.time()                print(f"  pushed {kept}", flush=True)    fh.close()    push()    print(f"\nkept    : {kept}")    print(f"dropped : {dropped}")    print(f"file    : {OUT_JSONL}")    print(f"merge it into the s2/s1 build by reading {OUT_JSONL} and adding src='teacher' rows")

In [ ]:
import torchMAX_SEQ_LENGTH = 1024      # p90 fits; 620-char cap in the data build keeps truncation at 0%DTYPE = torch.float16      # T4 is sm_75, no bf16print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)

## 2. Local tokenizerTwo upstream problems, fixed here rather than worked around:1. The abliterated base repo declares `tokenizer_class: TokenizersBackend`, a   **transformers 5.x** class. This notebook runs 4.57, so it cannot be resolved.2. **Neither** LFM repo embeds the chat template in `tokenizer_config.json`, so   transformers fetches it from the Hub - and that lookup 404s.So we build a local, self-contained tokenizer dir: 4.x-compatible class, templateembedded inline, and a real `model_max_length` (upstream ships `float('inf')` as aninteger). After this cell nothing touches the Hub for the tokenizer.Verified: vocab (64,400) and merges (63,683) are byte-identical to the abliteratedrepo, and this renders the same 29-token 2-turn string the data was built against.

In [ ]:
TOKENIZER_REPO = "LiquidAI/LFM2.5-1.2B-Instruct"TOK_DIR = "/kaggle/working/tok"os.makedirs(TOK_DIR, exist_ok=True)for fn in ["tokenizer.json", "tokenizer_config.json",           "chat_template.jinja", "special_tokens_map.json"]:    dst = os.path.join(TOK_DIR, fn)    if not os.path.exists(dst):        url = f"https://huggingface.co/{TOKENIZER_REPO}/resolve/main/{fn}"        import urllib.request        with urllib.request.urlopen(url, timeout=180) as r, open(dst, "wb") as f:            f.write(r.read())cfg_path = os.path.join(TOK_DIR, "tokenizer_config.json")cfg = json.load(open(cfg_path))cfg["chat_template"] = open(os.path.join(TOK_DIR, "chat_template.jinja")).read()cfg["model_max_length"] = MAX_SEQ_LENGTHcfg["tokenizer_class"] = "PreTrainedTokenizerFast"json.dump(cfg, open(cfg_path, "w"), ensure_ascii=False)from transformers import AutoTokenizer_chk = AutoTokenizer.from_pretrained(TOK_DIR)print("tokenizer ready |", len(_chk.get_vocab()), "tokens | max_len", _chk.model_max_length)print("probe:", repr(_chk.apply_chat_template(    [{"role": "user", "content": "kya haal hai?"},     {"role": "assistant", "content": " sab badhiya"}],    tokenize=False, add_generation_prompt=False)))

## 1d. Continued pre-training on raw Hindi (CPT)Liquid's own language-adaptation recipe is two stages, and we were only running thesecond one. From `cpt_translation_with_unsloth.ipynb`:> "perform **continued pre-training** on Korean Wikipedia data, **followed by**> instruction fine-tuning on Korean translation examples. **This approach is ideal> for adapting models to specific languages**"Their mechanism for the new script is in the LoRA call:> "We now add `embed_tokens` and `lm_head` to allow the model to learn **out of> distribution data**"with `embedding_learning_rate` set 2-10x below `learning_rate`, and`use_rslora=True` (True for CPT, False for SFT - the two notebooks differ on thisdeliberately).Session 2 previously went straight to SFT on 7,185 instruction pairs, so the modelhad never seen a single raw-text pass over Hindi. This stage supplies it, fromHindi Wikipedia, in exactly their format.**Why not `LiquidAI/LFM2.5-1.2B-Base`** as the CPT starting point, as the notebookuses: that model is not abliterated, and the abliteration is the reason for thiswhole exercise. Chaining CPT from the abliterated repo keeps it. Everything else inthis cell follows the vendor cell for cell.

In [ ]:
# Liquid's stage 1. Set RUN_CPT = False to skip and go straight to SFT.RUN_CPT    = TrueCPT_REPO   = f"{HF_USER}/lfm25-1.2b-cpt-hi"   # the SFT session 2 then chains fromCPT_STEPS  = 300CPT_LR     = 5e-5          # Liquid cpt_text_completion / cpt_translationCPT_EMB_LR = 1e-5          # "2 to 10x smaller than learning_rate"CPT_R      = 128           # "Add for continual pretraining" uses 128, SFT uses 16CPT_ALPHA  = 32CPT_DATA, CPT_CFG = "wikimedia/wikipedia", "20231101.hi"if not RUN_CPT:    print("RUN_CPT=False - SFT chains from BASE_REPO unchanged")else:    # Liquid sets this in every CPT notebook: CCE (cut cross entropy) is unsupported    # for continued pretraining. Must be set before the trainer is built.    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"    from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments    from transformers import AutoTokenizer    from datasets import load_dataset    # Same tokenizer dir the SFT stage uses. Loaded here because `tokenizer` does not    # exist yet - it is created by the model+LoRA cell further down.    cpt_tok = AutoTokenizer.from_pretrained(TOK_DIR)    assert cpt_tok.eos_token, "no eos_token; CPT would never learn to stop"    # --- data. Same prompt shape as the notebook's Korean Wikipedia cell, translated.    HI_PROMPT = "विकिपीडिया लेख\n### शीर्षक: {}\n\n### लेख:\n{}"    EOS_TOKEN = cpt_tok.eos_token   # vendor: "Must add EOS_TOKEN, otherwise your                                    # generation will go on forever!"    def cpt_format(examples):        # vendor appends EOS_TOKEN to every example        return {"text": [HI_PROMPT.format(t, b) + EOS_TOKEN                         for t, b in zip(examples["title"], examples["text"])]}    cpt_ds = load_dataset(CPT_DATA, CPT_CFG, split="train")    cpt_ds = cpt_ds.train_test_split(train_size=0.01, seed=3407)["train"]  # as the notebook    cpt_ds = cpt_ds.map(cpt_format, batched=True, remove_columns=cpt_ds.column_names)    print(f"CPT: {len(cpt_ds):,} Hindi Wikipedia articles, eos={EOS_TOKEN!r}")    # --- model. r=128 needs the cap raised with it.    cpt_model, _ = FastLanguageModel.from_pretrained(        model_name = BASE_REPO, tokenizer_name = TOK_DIR,        max_seq_length = MAX_SEQ_LENGTH, dtype = DTYPE,        load_in_4bit = False, max_lora_rank = CPT_R,    )    # vendor: "We now add embed_tokens and lm_head to allow the model to learn out of    # distribution data" - this is the whole mechanism for a new script.    CPT_TARGETS = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3",                   "embed_tokens", "lm_head"]    _names = [n for n, _ in cpt_model.named_modules()]    _missing = [m for m in CPT_TARGETS                if not any(n == m or n.endswith("." + m) for n in _names)]    if _missing:        raise RuntimeError(f"CPT target modules not present: {_missing}")    print("CPT target modules verified:", CPT_TARGETS)    cpt_model = FastLanguageModel.get_peft_model(        cpt_model, r = CPT_R, lora_alpha = CPT_ALPHA, lora_dropout = 0, bias = "none",        target_modules = CPT_TARGETS,        use_gradient_checkpointing = "unsloth",        random_state = 3407,        use_rslora = True,   # True for CPT, False for SFT - the vendor differs deliberately    )    _n = sum(p.numel() for p in cpt_model.parameters() if p.requires_grad)    _a = sum(p.numel() for p in cpt_model.parameters())    print(f"CPT trainable {_n/1e6:.1f}M / {_a/1e9:.2f}B ({100*_n/_a:.2f}%)")    if _n == 0:        raise RuntimeError("CPT LoRA attached to zero parameters - check CPT_TARGETS")    cpt_trainer = UnslothTrainer(        model = cpt_model, tokenizer = cpt_tok,        train_dataset = cpt_ds, eval_dataset = None,        dataset_text_field = "text", max_seq_length = MAX_SEQ_LENGTH,        dataset_num_proc = 4,        args = UnslothTrainingArguments(            per_device_train_batch_size = 2,            gradient_accumulation_steps = 8,        # effective 16, as the CPT notebook            max_steps = CPT_STEPS, warmup_steps = 10,            learning_rate = CPT_LR,            embedding_learning_rate = CPT_EMB_LR,   # exists only because of CPT            logging_steps = max(5, CPT_STEPS // 60),            optim = "adamw_8bit", weight_decay = 0.001,            lr_scheduler_type = "linear",            seed = 3407, output_dir = "cpt-out",            save_strategy = "no", report_to = [],        ),    )    # No response masking: CPT trains on the whole document, which is the point of it.    cpt_trainer.add_callback(TimeCheckpoint(cpt_model, CPT_REPO,                                           every_s=CHECKPOINT_EVERY_S, tokenizer=cpt_tok))    print(f"CPT ready - {CPT_STEPS} steps -> {CPT_REPO}")    cpt_model.generation_config.max_length = None    cpt_model.generation_config.max_new_tokens = None    cpt_model.push_to_hub_merged(CPT_REPO, tokenizer=cpt_tok, save_method="merged_16bit")    cpt_model.push_to_hub(CPT_REPO, tokenizer=cpt_tok)   # LoRA too, so it is resumable    print("CPT done ->", f"https://huggingface.co/{CPT_REPO}")    print(f"set SESSION = 2 and BASE_REPO = '{CPT_REPO}' to chain SFT from it")

## 3. Model + LoRALFM's MLP is `w1/w2/w3` and it has an `in_proj` - not the usual LLaMA names.`embed_tokens`/`lm_head` are deliberately NOT trained: the tokenizer is unchanged,so there is no new vocabulary to learn. Devanagari rides on the existing byte tokens.`use_gradient_checkpointing="unsloth"` stays on - session 2's sequences are 3-9xlonger and the spare VRAM session 1 shows is not really spare.

In [ ]:
# Weights from the abliterated repo; tokenizer from the local dir built above.model, tokenizer = FastLanguageModel.from_pretrained(    model_name = BASE_REPO, tokenizer_name = TOK_DIR,    max_seq_length = MAX_SEQ_LENGTH,    dtype = DTYPE, load_in_4bit = False, max_lora_rank = 32,)# Guard BEFORE attaching: named_modules() returns dotted paths like# "model.layers.0.self_attn.q_proj", so a bare "q_proj" is never an exact member.# A wrong name attaches nothing and the run trains happily on zero parameters.TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"]_names = [n for n, _ in model.named_modules()]_missing = [m for m in TARGET_MODULES if not any(n == m or n.endswith("." + m) for n in _names)]if _missing:    raise RuntimeError(f"Target modules not present in this model: {_missing}")print("target modules verified:", TARGET_MODULES)model = FastLanguageModel.get_peft_model(    model, r = 32, lora_alpha = 32, lora_dropout = 0, bias = "none",    target_modules = TARGET_MODULES,    use_gradient_checkpointing = "unsloth",    # use_rslora scales by alpha/sqrt(r); plain LoRA scales by alpha/r. At r=alpha=32    # that is 5.66 vs 1.00, so True was training at 5.66x the intended adapter scale.    # Liquid's own SFT notebook uses False; True is for the CPT/pretraining recipe.    random_state = 3407, use_rslora = False,)n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)n_all = sum(p.numel() for p in model.parameters())print(f"trainable {n_train/1e6:.1f}M / {n_all/1e9:.2f}B ({100*n_train/n_all:.2f}%)")if n_train == 0:    raise RuntimeError("LoRA attached to zero parameters - check TARGET_MODULES")

## 4. Chat template`apply_chat_template` already adds `<|startoftext|>` and terminates with`<|im_end|>`. Do not append an EOS yourself - most tutorials do and itdouble-terminates.`train_on_responses_only` masks the user turn so loss is computed on the assistantresponse only. It is applied in the trainer cell below, and the cell prints the-100 ratio so you can see it took effect rather than trusting this sentence.

In [ ]:
from datasets import Datasetdef to_text(batch):    convs = [[{"role": "user", "content": i}, {"role": "assistant", "content": r}]             for i, r in zip(batch["instruction"], batch["response"])]    # apply_chat_template prepends <|startoftext|>. Strip it explicitly, as Liquid's    # unsloth-sft-lfm2.5.py does, rather than relying on Unsloth's auto-fix, which only    # announces itself in a log line.    return {"text": [tokenizer.apply_chat_template(c, tokenize=False,                                                   add_generation_prompt=False)                            .removeprefix(tokenizer.bos_token or "")                     for c in convs]}ds = Dataset.from_list(rows).map(to_text, batched=True, remove_columns=list(rows[0].keys()))split = ds.train_test_split(test_size=0.02, seed=3407)print("train", len(split["train"]), "eval", len(split["test"]))print(repr(split["train"][0]["text"][:180]))

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArgumentstrainer = UnslothTrainer(    model = model, tokenizer = tokenizer,    train_dataset = split["train"], eval_dataset = split["test"],    dataset_text_field = "text", max_seq_length = MAX_SEQ_LENGTH,    args = UnslothTrainingArguments(        per_device_train_batch_size = 8,        gradient_accumulation_steps = 1,        max_steps = MAX_STEPS,        warmup_steps = 30,        learning_rate = LEARNING_RATE,        lr_scheduler_type = "linear",        optim = "adamw_8bit",        weight_decay = 0.01,        # 4 loss readings across 690 steps looked like "spiking" and invited the        # wrong fix. 69 points show the real shape. Eval every max_steps//5 as        # Liquid's unsloth-sft-lfm2.5.py does, so session 2's 400 steps gets 5        # points instead of 2.        logging_steps = max(5, MAX_STEPS // 60),        eval_strategy = "steps", eval_steps = max(20, MAX_STEPS // 5),        seed = 3407, output_dir = "hf-out",        save_strategy = "no",          # TimeCheckpoint owns saving        report_to = [],    ),)# Mask the user turn so loss is computed on the ASSISTANT RESPONSE ONLY. This was# documented in the chat-template cell as if it were happening; it never was. Without# it the model is trained to emit the question as well as the answer, which is exactly# the short-prompt collapse we spent four sessions chasing - given "kya haal hai?"# the model has been taught to produce a user turn.from unsloth.chat_templates import train_on_responses_onlytrainer = train_on_responses_only(    trainer,    instruction_part = "<|im_start|>user\n",    response_part    = "<|im_start|>assistant\n",)_lab = trainer.train_dataset[0]["labels"]_masked = sum(1 for x in _lab if x == -100)print(f"label masking: {_masked}/{len(_lab)} tokens are -100 "      f"({100*_masked/len(_lab):.0f}% ignored)")if _masked == 0:    raise RuntimeError("train_on_responses_only did not mask anything")trainer.add_callback(TimeCheckpoint(model, OUT_REPO,                                   every_s=CHECKPOINT_EVERY_S, tokenizer=tokenizer))print("trainer ready - checkpoints push every", CHECKPOINT_EVERY_S, "s")

In [ ]:
stats = trainer.train()print(stats.metrics)# The full curve, so the run is judged on 69 points and not on 4._curve = [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h]print(f"\ntrain loss ({len(_curve)} points):")for a in range(0, len(_curve), 8):    print("   " + "  ".join(f"{s_:>4}:{l:.3f}" for s_, l in _curve[a:a+8]))# Rising eval loss = the LoRA memorising this session's slice rather than learning a# language shift. Sessions 2/3 chain from whatever gets pushed, so catching it here is# far cheaper than discovering it as a broken model later.evals = [(h["step"], round(h["eval_loss"], 4))         for h in trainer.state.log_history if "eval_loss" in h]if evals:    print("eval_loss by step:", evals)    # Liquid's own generalisation gate, from unsloth-sft-lfm2.5.py: an eval/train ratio    # above 1.5 means the adapter is memorising rather than learning.    _tl = [h["loss"] for h in trainer.state.log_history if "loss" in h]    if _tl:        _ratio = evals[-1][1] / _tl[-1]        print(f"eval/train ratio: {_ratio:.2f}"              + ("  - generalises well" if _ratio <= 1.5 else "  - OVERFITTING"))    if len(evals) >= 2 and evals[-1][1] > evals[0][1]:        print("WARNING: eval loss rose - overfitting. Lower MAX_STEPS before the "              "next session.")    else:        print("eval loss stable or falling - OK to chain.")

## 5. LicenceThe base is under **Liquid Open License v1.0** (`lfm1.0`), which has 16redistribution clauses - the text must travel with the weights, not just themodel card name.

In [ ]:
from huggingface_hub import upload_fileimport urllib.request_url = "https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct/resolve/main/LICENSE"with urllib.request.urlopen(_url, timeout=180) as r:    _lic = r.read()os.makedirs("hf-out", exist_ok=True)with open("hf-out/LICENSE", "wb") as f:    f.write(_lic)upload_file(path_or_fileobj="hf-out/LICENSE", path_in_repo="LICENSE", repo_id=OUT_REPO)print(f"LICENSE pushed ({len(_lic):,} bytes)")

## 6. Merge -> GGUF -> HubThis is the long pole for the 30-minute budget. Even if it never runs, the`ckpt-*` adapters are already on the Hub every 5 minutes.

In [ ]:
import os, shutil# Unsloth writes the intermediate f16 conversion to the CURRENT WORKING DIRECTORY and# then moves it to the export dir. If those are on different filesystems the move is a# 4.2 GB copy, and the preflight check reads the cwd's free space. On Kaggle# /kaggle/working holds only ~20 GB, so pulling a teacher into it can leave 0.9 GB free# and the export dies even though /tmp has over a terabyte. chdir first so the# conversion is written where the space is.shutil.rmtree("/kaggle/working/teacher", ignore_errors=True)   # 16.4 GB teacher, if presentos.chdir("/tmp")_free = shutil.disk_usage("/tmp").free / 2**30print(f"cwd {os.getcwd()}  {_free:.0f} GB free")if _free < 10:    raise RuntimeError(f"only {_free:.0f} GB on /tmp - not enough for the f16 pass")# Unsloth resets generation_config when it switches modes, leaving max_length baked# in. That default rides into the merged model and then into the GGUF you deploy,# so clear it BEFORE save_pretrained_gguf.model.generation_config.max_length = Nonemodel.generation_config.max_new_tokens = Nonep = model.save_pretrained_gguf("hf-out", tokenizer, quantization_method = "q4_k_m")print("gguf at:", p)model.push_to_hub_gguf(OUT_REPO, tokenizer, quantization_method = "q4_k_m")model.push_to_hub(OUT_REPO, tokenizer)      # the next session loads thisprint("done ->", f"https://huggingface.co/{OUT_REPO}")

## 7. Back on the server```hf download <you>/lfm25-1.2b-bilingual-s2 --include "*Q4_K_M*" \  --local-dir ~/finetune-models/s2          # NOT models/@models - see note```Always download to a per-session dir. `save_pretrained_gguf` names the file afterthe **base** model, so s1/s2/s3 all produce `...Uncensored.Q4_K_M.gguf` and theyclobber each other in a shared directory.Serve it and gate it:```nohup ~/llama.cpp/build/bin/llama-server \  -m ~/finetune-models/s2/*Q4_K_M*.gguf -c 4096 --threads 2 \  --host 127.0.0.1 --port 8081 --no-webui &python3.13 /tmp/hindi.py 8081```**Gates, in priority order:**1. `hindi 5/5` with **zero mojibake** - the whole point. Baseline before session 2   is `0/5`, all romanized, zero mojibake: the model can *read* Devanagari but had   never seen a Devanagari *reply*, so it answered in Latin script.2. `hinglish 5/5` - must not regress3. English intact4. **Refusal compliance** - the abliteration lives in the weights you just   finetuned; a language LoRA can wash it out. This is the likeliest failure.